In [1]:
!pip install torch torchvision torchaudio torchao --index-url https://download.pytorch.org/whl/cu121

!pip install -q transformers==5.6.2
!pip install -q accelerate==1.13.0
!pip install -q peft>=0.19.1
!pip install -q bitsandbytes>=0.49.2
!pip install -q trl==0.19.0
!pip install -q "datasets<4.0.0"
!pip install -q evaluate>=0.4.6
!pip install -q scikit-learn stanza

Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 41.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 33.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.9 MB/s et

In [2]:
!pip install --upgrade gdown

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.1
    Uninstalling gdown-5.2.1:
      Successfully uninstalled gdown-5.2.1


In [3]:
import torch
import gc
import os
import json
import pandas as pd
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from datasets import Dataset, load_dataset

In [4]:
# Memaksa PyTorch hanya melihat dan menggunakan GPU pertama
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
os.makedirs("/kaggle/working/C0_Baseline/", exist_ok=True)

In [6]:
import torch

print("=" * 50)
print("GPU CHECK")
print("=" * 50)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM")
    print(f"\nTotal GPU: {torch.cuda.device_count()}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("❌ GPU tidak ditemukan! Pastikan Kaggle Accelerator aktif.")
    print("Pergi ke: Settings → Accelerator → GPU T4 x2")

GPU CHECK
GPU 0: Tesla T4 — 14.6 GB VRAM

Total GPU: 1
CUDA version: 12.1


In [7]:
CONFIG = {
    "model_name": "Qwen/Qwen2-1.5B",
    "output_dir": "/kaggle/working/",
    # "output_dir": "/content/drive/MyDrive/Riset_QLoRA/qwen-lora-adapter",
    "wiki_dataset_dir": "/content/drive/MyDrive/Riset_QLoRA/wiki_budaya_5100.csv",
    # QLoRA Params
    "lora_r": 32,             # Rank lebih besar untuk kompensasi model kecil (default = 64)
    "lora_alpha": 128,
    "target_modules": "all-linear",

    # Training Params
    "max_seq_length": 512,
    "batch_size": 2,
    "grad_acc_steps": 8,      # Effective batch = 16
    "lr": 2e-5,               # LR rendah untuk cegah forgetting
    "epochs": 3, # default for this xp = 3

    # Dataset Subset
    # "train_samples": 20000,
    # "eval_limit": 300,        # Limit evaluasi per task agar cepat
    "train_samples": 8000,
    "eval_limit": 150,
}

print("✅ Konfigurasi berhasil dimuat")
print(f"Model: {CONFIG['model_name']}")
print(f"Training samples: {CONFIG['train_samples']}")

✅ Konfigurasi berhasil dimuat
Model: Qwen/Qwen2-1.5B
Training samples: 8000


In [8]:
# ---- Dataset Evaluasi (Zero-shot / Post-training) ----
print("Memuat dataset eval...")
eval_tasks = {}

try:
    eval_tasks["copal"] = load_dataset("haryoaw/COPAL", split="test", trust_remote_code=True)
    print(f"✅ COPAL-ID: {len(eval_tasks['copal'])} sampel")
except Exception as e:
  print(f"❌ COPAL-ID gagal diload: {e}")

try:
    smsa_df = pd.read_csv("/kaggle/input/datasets/vorstellenze/smsa-dataset/smsa_dataset_reconstructed.csv")
    eval_tasks['smsa'] = Dataset.from_pandas(smsa_df)
    print(f"✅ smsa: {len(eval_tasks['smsa'])} sampel")
except Exception as e:
    print(f"❌ smsa gagal diload: {e}")

try:
    eval_tasks["indonli"] = load_dataset("afaji/indonli", split="test_expert", trust_remote_code=True)
    print(f"✅ IndoNLI: {len(eval_tasks['indonli'])} sampel")
except Exception as e:
  print(f"❌ IndoNLI gagal diload: {e}")

try:
    eval_tasks["indoculture"] = load_dataset("indolem/IndoCulture", split="test", trust_remote_code=True)
    print(f"✅ IndoCulture: {len(eval_tasks['indoculture'])} sampel")
except Exception as e:
  print(f"❌ IndoCulture gagal diload: {e}")

try:
    # Menggunakan TyDiQA Gold Passage khusus split bahasa Indonesia
    qa_full = load_dataset("tydiqa", "secondary_task", split="validation")

    # Filter hanya untuk bahasa Indonesia
    eval_tasks["qa"] = qa_full.filter(lambda x: x["id"].startswith("indonesian"))
    print(f"✅ QA (TyDiQA Official): {len(eval_tasks['qa'])} sampel")
except Exception as e:
    print(f"❌ QA gagal diload: {e}")

Memuat dataset eval...


README.md: 0.00B [00:00, ?B/s]

test_copal.csv: 0.00B [00:00, ?B/s]

test_copal_colloquial.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/559 [00:00<?, ? examples/s]

...

secondary_task/validation-00000-of-00001(…):   0%|          | 0.00/2.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49881 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5077 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5077 [00:00<?, ? examples/s]

✅ QA (TyDiQA Official): 565 sampel


In [9]:
import json
import re
import os
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


def extract_hybrid_prediction_final(task_name, gen_text):
    gen_text = str(gen_text).strip().upper()

    if task_name in ["copal", "indoculture"]:
        # Pattern 1: Huruf A/B/C di awal
        match = re.search(r'^([A-C])[\.\s\(\:]', gen_text)
        if match: return match.group(1)

        # Pattern 2: "JAWABAN: X"
        match = re.search(r'JAWABAN[\s\:]+([A-C])', gen_text)
        if match: return match.group(1)

        # Pattern 3: "JAWABAN (X)" 
        match = re.search(r'JAWABAN\s*\(([A-C])\)', gen_text)
        if match: return match.group(1)

        # Pattern 4: Huruf tunggal
        if len(gen_text) == 1 and gen_text in ['A', 'B', 'C']:
            return gen_text

        # Pattern 5: "ADALAH X"
        for label in ["A", "B", "C"]:
            if f"ADALAH {label}" in gen_text or f"PILIHAN {label}" in gen_text:
                return label

        # Pattern 6: Fallback dengan boundary separator
        match = re.search(r'\b([A-C])\b[\.\s\(\:\|]', gen_text)
        if match: return match.group(1)

    elif task_name == "indonli":
        if "SESUAI" in gen_text or "ENTAILMENT" in gen_text: return "SESUAI"
        if "BERTENTANGAN" in gen_text or "CONTRADICTION" in gen_text: return "BERTENTANGAN"
        if "NETRAL" in gen_text or "NEUTRAL" in gen_text: return "NETRAL"

    elif task_name == "smsa":
        if "POSITIF" in gen_text or "POSITIVE" in gen_text: return "POSITIF"
        if "NEGATIF" in gen_text or "NEGATIVE" in gen_text: return "NEGATIF"
        if "NETRAL" in gen_text or "NEUTRAL" in gen_text: return "NETRAL"

    elif task_name == "qa":
        clean_text = gen_text.split('\n')[0].strip()
        match = re.search(r'^[A-C]\s*[\(\.\:]\s*(.*)', clean_text)
        if match: return match.group(1).replace(")", "").strip()
        return clean_text

    return "UNKNOWN"

In [10]:
print("="*70)
print("🚀 MEMULAI RE-EVALUASI BASELINE (C0) - STRICT METRICS")
print("="*70)

# ============================================================
# 1. FUNGSI EVALUASI BASELINE MURNI (Zero-Shot)
# ============================================================
def evaluate_baseline_pure(model, tokenizer, dataset, task_name, batch_size=8):
    model.eval()
    results = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Eval C0 {task_name}"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = []

        for s in batch:
            if task_name == "copal":
                q_type = str(s.get("question","")).lower()
                pertanyaan = "Apa penyebab dari situasi tersebut?" if q_type == "cause" else \
                             "Apa akibat dari situasi tersebut?" if q_type == "effect" else \
                             "Manakah pilihan yang paling masuk akal?"
                p = f"Situasi: {s['premise']}\nPertanyaan: {pertanyaan}\nA. {s['choice1']}\nB. {s['choice2']}\nJawaban (A/B):"

            elif task_name == "smsa":
                p = f"Tentukan sentimen teks berikut (positif/negatif/netral):\n'{s['text']}'\nSentimen:"

            elif task_name == "indonli":
                p = f"Premis: {s['premise']}\nHipotesis: {s['hypothesis']}\nHubungan (Entailment/Contradiction/Neutral):"

            elif task_name == "indoculture":
                options_list = s.get("options", [])
                options_text = "\n".join(options_list) if isinstance(options_list, list) else str(options_list)
                p = f"Konteks: {s['context']}\nPilihan:\n{options_text}\nJawaban (A/B/C):"

            elif task_name == "qa":
                passage_text = " ".join(s.get("passage",[])).replace(" ,",",").replace(" .",".")
                question_text = " ".join(s.get("question",[])).replace(" ,",",").replace(" ?","?")
                p = f"Teks: {passage_text}\nPertanyaan: {question_text}\nJawaban:"

            # ChatML Wrapper (sesuai Qwen default)
            msg = [{"role": "user", "content": p}]
            prompts.append(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)

        for j, out in enumerate(outputs):
            gen_text = tokenizer.decode(out[inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
            results.append({"pred": gen_text})

    return results



🚀 MEMULAI RE-EVALUASI BASELINE (C0) - STRICT METRICS


In [11]:
# ============================================================
# 2. LOAD BASE MODEL (Tanpa LoRA/Adapter)
# ============================================================
print("⏳ Memuat model Qwen2 1.5B murni...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2-1.5B")
tokenizer_base.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-1.5B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("✅ Base model siap!")



⏳ Memuat model Qwen2 1.5B murni...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ Base model siap!


In [12]:
# ============================================================
# 3. EKSEKUSI EVALUASI & KALKULASI METRIK KETAT
# ============================================================
baseline_results = []
os.makedirs("/kaggle/working/C0_Baseline", exist_ok=True)

# Asumsi eval_tasks dan extract_hybrid_prediction_final sudah ada di memori
for task_name, dataset in eval_tasks.items():
    print(f"\nMengeksekusi Task: {task_name.upper()} ({len(dataset)} sampel)")

    raw_results = evaluate_baseline_pure(base_model, tokenizer_base, dataset, task_name, batch_size=1)

    y_true, y_pred = [], []
    format_errors = 0
    detailed_logs = []

    for i, res in enumerate(raw_results):
        raw_pred = res["pred"]
        clean_pred = extract_hybrid_prediction_final(task_name, raw_pred) # Panggil Parser V3
        
        # Ekstrak GT
        gt_label = str(dataset[i].get("label", dataset[i].get("answer", dataset[i].get("target", "")))).strip().upper()
        
        # Normalisasi label untuk metrik
        if task_name == "indoculture" and gt_label in ["0", "1", "2"]:
            gt_label = {"0": "A", "1": "B", "2": "C"}[gt_label]
        elif task_name == "copal" and gt_label in ["0", "1"]:
            gt_label = {"0": "A", "1": "B"}[gt_label]
        elif task_name == "indonli":
            gt_label = {"0": "SESUAI", "1": "NETRAL", "2": "BERTENTANGAN"}.get(gt_label, gt_label)
        elif task_name == "smsa":
            gt_label = {"0": "POSITIF", "1": "NETRAL", "2": "NEGATIF"}.get(gt_label, gt_label)

        y_true.append(gt_label)
        y_pred.append(clean_pred)
        
        if clean_pred == "UNKNOWN":
            format_errors += 1

        detailed_logs.append({
            "id": i,
            "ground_truth": gt_label,
            "prediction": clean_pred,
            "raw_output": raw_pred,
        })

    # KALKULASI METRIK (SANGAT KETAT: UNKNOWN = SALAH)
    acc = accuracy_score(y_true, y_pred)
    labels_present = list(set(y_true))
    
    if task_name == "qa":
        f1 = 0.0 # QA diukur dengan EM/F1 token base nanti
    else:
        f1 = f1_score(y_true, y_pred, labels=labels_present, average="macro", zero_division=0)

    print(f"   Acc: {acc*100:05.2f}% | F1: {f1*100:05.2f}% | Errors: {format_errors}/{len(dataset)}")

    baseline_results.append({
        "Condition": "C0_Baseline",
        "Task": task_name.upper(),
        "Accuracy": round(acc, 4),
        "Macro_F1": round(f1, 4),
        "Format_Errors": format_errors,
        "Total_Samples": len(dataset),
    })

    # Simpan log json agar tidak hilang lagi
    with open(f"/kaggle/working/C0_Baseline/{task_name}_log.json", "w", encoding="utf-8") as f:
        json.dump(detailed_logs, f, ensure_ascii=False, indent=2)

# Export tabel
df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv("/kaggle/working/C0_baseline_results.csv", index=False)
print("\n✅ Evaluasi Baseline selesai dan disimpan!")

# Bersihkan VRAM
del base_model
del tokenizer_base
gc.collect()
torch.cuda.empty_cache()
print("✅ VRAM berhasil dibersihkan. Memori GPU lega kembali.")


Mengeksekusi Task: COPAL (559 sampel)


Eval C0 copal: 100%|██████████| 559/559 [07:26<00:00,  1.25it/s]


   Acc: 00.18% | F1: 00.36% | Errors: 558/559

Mengeksekusi Task: SMSA (500 sampel)


Eval C0 smsa: 100%|██████████| 500/500 [06:40<00:00,  1.25it/s]


   Acc: 76.40% | F1: 57.53% | Errors: 26/500

Mengeksekusi Task: INDONLI (2984 sampel)


...

   Acc: 00.94% | F1: 01.69% | Errors: 2913/2984

Mengeksekusi Task: INDOCULTURE (2429 sampel)


Eval C0 indoculture: 100%|██████████| 2429/2429 [32:53<00:00,  1.23it/s]


   Acc: 01.56% | F1: 02.57% | Errors: 2308/2429

Mengeksekusi Task: QA (565 sampel)


Eval C0 qa: 100%|██████████| 565/565 [07:17<00:00,  1.29it/s]


   Acc: 00.00% | F1: 00.00% | Errors: 0/565

✅ Evaluasi Baseline selesai dan disimpan!
✅ VRAM berhasil dibersihkan. Memori GPU lega kembali.
